# Week 13 — Companion Notebook: Customer Discovery & Scoping

**The Meridian Engagement · the deep reference**

This notebook is the depth behind the lean slides. It is **pure-Python / stdlib** — no external services, no API keys, no installs. It runs top to bottom with no edits.

Every named term from the slides is explained here, and every M0 artifact has a runnable template + a self-check helper:

1. **Why FDEs exist** + the FDE lifecycle (Scope -> Build -> Deploy -> Operate)
2. **`DECISIONS.md`** template + a helper that writes valid entries
3. **Eval-anchored success criteria** — a rubric + `score_criterion()` that scores vague vs measurable
4. **Stakeholder / RACI builder**
5. **Integration-inventory** template
6. **The 911-response-time decomposition drill** — worked example
7. A `score_process_map()` self-check for the discovery-workshop artifact

> Run `Cell -> Run All`. Everything prints inline. Nothing writes outside this folder except the demo `DECISIONS.md` / charter files, which land in a local `m0_charter/` directory you can inspect or delete.

## 0. Setup

Stdlib only. We define a tiny assert helper so the notebook self-verifies as it runs — if a cell prints `OK`, that section works.

In [ ]:
import os, json, re, textwrap, datetime
from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import Optional

WORKDIR = 'm0_charter'
os.makedirs(WORKDIR, exist_ok=True)

def show(title, body=''):
    bar = '=' * len(title)
    print(title); print(bar)
    if body:
        print(body)
    print()

def expect(cond, msg):
    """Lightweight assert that prints OK instead of crashing on success."""
    assert cond, 'FAILED: ' + msg
    print('OK -', msg)

print('Setup complete. Working dir:', os.path.abspath(WORKDIR))

## 1. Why FDEs exist, and the FDE lifecycle

**The 95% problem.** MIT's Project NANDA report *The GenAI Divide* (2025) found roughly **95% of organizations get zero measurable return** from their GenAI pilots; only about **5% of integrated pilots** extract real value. The bottleneck it measured is not model capability. It is **deployment**: the wrong problem, no integration with real systems, and no agreed definition of success.

A **Forward-Deployed Engineer (FDE)** is the person who closes that gap by embedding with the client. The industry has industrialized the role: frontier labs run dedicated deployment organizations and enterprise-services ventures built around FDEs — a structural bet that deployment, not the model, is the constraint. (Specific deals, sizes, and dates rot fast; verify before quoting any.)

**The FDE lifecycle** is the spine of this whole course. Each stage answers one question:

In [ ]:
FDE_LIFECYCLE = [
    ('Scope',   'What is the real problem, measurably?',          'W13 (this week)'),
    ('Sell',    'What will it cost, and what are we buying?',      'W14'),
    ('Build',   'Does it work on real cases?',                     'W15'),
    ('Integrate','Can it see the client systems, safely?',         'W16'),
    ('Operate', 'Can we prove it, then hand it over?',             'W17'),
]

show('THE FDE LIFECYCLE')
for stage, question, week in FDE_LIFECYCLE:
    print(f'  {stage:<10} {question:<48} [{week}]')
print()
print('Today we are entirely in SCOPE. The discipline is resisting the jump to BUILD.')
expect(FDE_LIFECYCLE[0][0] == 'Scope', 'lifecycle starts at Scope')

## 2. `DECISIONS.md` — decision discipline

**`DECISIONS.md`** is a lightweight Architecture Decision Record (ADR) log kept in the repo. One entry per real decision. The format:

- **Decision** — what you chose
- **Why** — the reasoning (this is the field that evaporates from memory in two weeks)
- **Alternatives** — what you considered and rejected, and why
- **Owner** — who made the call
- **Revisit** — the trigger that should reopen the decision

In week five, when a stakeholder says *"that's not what we agreed,"* this file is the difference between a renewal and a fight. The `Why` and `Alternatives` fields carry the value; an entry without them is a changelog, not a decision log.

In [ ]:
@dataclass
class Decision:
    decision: str
    why: str
    alternatives: str
    owner: str
    revisit: str
    date: str = field(default_factory=lambda: datetime.date.today().isoformat())

    def to_markdown(self) -> str:
        return textwrap.dedent(f'''\
        ## {self.date} - {self.decision}
        - **Decision:** {self.decision}
        - **Why:** {self.why}
        - **Alternatives considered:** {self.alternatives}
        - **Owner:** {self.owner}
        - **Revisit when:** {self.revisit}
        ''').strip()

def write_decisions_md(decisions, path):
    header = '# DECISIONS.md\n\nEngagement decision log (ADR-lite). Newest first.\n'
    body = '\n\n'.join(d.to_markdown() for d in reversed(decisions))
    with open(path, 'w') as f:
        f.write(header + '\n' + body + '\n')
    return path

def lint_decision(d: Decision):
    """Returns a list of problems. Empty list == a good entry."""
    problems = []
    if len(d.why.split()) < 4:
        problems.append('Why is too thin - explain the reasoning, not just the choice')
    if d.alternatives.strip().lower() in ('', 'n/a', 'none', 'na'):
        problems.append('Alternatives is empty - every real decision had a road not taken')
    if not d.owner.strip():
        problems.append('Owner missing - decisions need an accountable name')
    return problems

In [ ]:
# The opening M0 entry: why W13 is discovery-only.
d0 = Decision(
    decision='W13 is discovery only - no build until success criteria are agreed',
    why='Nobody at Meridian can define success yet. Building now means building the '
        'wrong thing fast and burning trust with a hostile ops head.',
    alternatives='Start a prototype to show momentum (rejected - momentum toward an '
                 'undefined target is just risk we will pay for in week five).',
    owner='Forgepoint FDE team',
    revisit='After M0 charter sign-off',
)

d1 = Decision(
    decision='Scope the pilot to Type-A (fraud) disputes first',
    why='Type-A has the cleaner Reg E clock and the clearest auto-triage signal; it '
        'lets us prove value before touching the messier Type-B merchant cases.',
    alternatives='Do both types at once (rejected - doubles the eval surface and the '
                 'RBAC complexity before we have proven anything).',
    owner='Forgepoint FDE team',
    revisit='After M2 hits >=80% auto-triage on Type-A',
)

decisions = [d0, d1]
path = write_decisions_md(decisions, os.path.join(WORKDIR, 'DECISIONS.md'))
show('WROTE ' + path)
print(open(path).read())

for d in decisions:
    expect(lint_decision(d) == [], f'decision passes lint: {d.decision[:40]}...')

## 3. Eval-anchored success criteria

**Eval-anchored** means success is written as a **measurable criterion you could test on day one and at handover** — not an adjective. The move is to turn every executive adjective ("faster," "better," "accurate") into `[metric] [threshold] [population]`.

Why scope evals **at discovery, not at the end**: "what does success mean" is a *client* conversation, and the client is in the room now. Defer it to week 17 and you'll define success alone, to match whatever you happened to build. This is the seed of the W17 eval suite — and eval discipline is the single biggest technical differentiator the FDE interview filters on.

### A scorer for vague-vs-measurable criteria

`score_criterion()` rates a candidate success criterion. It checks for the three things a testable criterion needs and flags the adjectives that signal a vague one.

In [ ]:
VAGUE_WORDS = {
    'faster','better','accurate','accurately','good','great','improve','improved',
    'efficient','efficiently','quickly','quick','robust','seamless','smart','smarter',
    'easy','easier','reliable','high-quality','optimal','significantly','a lot','some',
}

# signals that a threshold / number is present
NUMBER_RE = re.compile(r'(\d+(?:\.\d+)?\s*%|\bzero\b|\b0\b|\d+(?:\.\d+)?\s*'
                       r'(?:min|minute|minutes|sec|second|seconds|hour|hours|day|days|'
                       r'cases|disputes|cents|dollars|\$))', re.I)
COMPARATOR_RE = re.compile(r'(>=|<=|>|<|=|\bat least\b|\bat most\b|\bno more than\b|'
                           r'\bunder\b|\bover\b|\bbelow\b|\babove\b|\bequal\b|'
                           r'\bexactly\b|\bwithin\b)', re.I)
# signals that a population / 'on what' is named
POPULATION_RE = re.compile(r'(\bon\b|\bof\b|\bper\b|\bType[- ]?[AB]\b|\bgolden\b|'
                           r'\bcases\b|\bdisputes\b|\bset\b|\bsubset\b|\bpopulation\b)',
                           re.I)

@dataclass
class CriterionScore:
    text: str
    has_number: bool
    has_comparator: bool
    has_population: bool
    vague_terms: tuple
    verdict: str
    notes: tuple

def score_criterion(text: str) -> CriterionScore:
    has_number = bool(NUMBER_RE.search(text))
    has_comparator = bool(COMPARATOR_RE.search(text))
    has_population = bool(POPULATION_RE.search(text))
    found_vague = tuple(w for w in VAGUE_WORDS if re.search(r'\b'+re.escape(w)+r'\b', text, re.I))
    notes = []
    if not has_number:
        notes.append('no measurable threshold (add a number: %, count, time, $)')
    if not has_comparator:
        notes.append('no comparator (>=, <=, zero, at least, within...)')
    if not has_population:
        notes.append('no population (on WHAT cases? Type-A? the 150-case golden set?)')
    if found_vague and not has_number:
        notes.append('relies on adjectives ' + str(list(found_vague)) + ' instead of a metric')
    # MEASURABLE requires a real threshold AND a named population.
    if has_number and has_population:
        verdict = 'MEASURABLE'
    elif has_number:
        verdict = 'PARTIAL (has a number, missing population)'
    else:
        verdict = 'VAGUE'
    return CriterionScore(text, has_number, has_comparator, has_population,
                          found_vague, verdict, tuple(notes))

print('score_criterion() defined.')

In [ ]:
examples = [
    'Make disputes faster',
    'Improve accuracy significantly',
    'Reduce average handle time',
    'Auto-triage at least 80% of Type-A disputes with >=95% policy-citation accuracy '
        'on the 150-case golden set',
    'Agent-caused Reg E clock breaches equal exactly 0 across all disputes',
    'Provisional-credit errors at most the current human baseline on Type-A cases',
]

show('SCORING VAGUE VS MEASURABLE CRITERIA')
for ex in examples:
    s = score_criterion(ex)
    print(f'[{s.verdict}]')
    print('  ' + ex)
    if s.notes:
        for n in s.notes:
            print('    - ' + n)
    print()

# self-checks: the rewrite IS measurable, the executive adjective is NOT
expect(score_criterion(examples[3]).verdict == 'MEASURABLE', 'good rewrite scores MEASURABLE')
expect(score_criterion(examples[0]).verdict == 'VAGUE', "'make disputes faster' scores VAGUE")
expect(score_criterion(examples[4]).verdict == 'MEASURABLE', 'zero-breach guardrail scores MEASURABLE')

### The eval-anchoring rubric

Use this to grade your own M0 success criteria. A criterion passes the **floor** if it is MEASURABLE; it earns the **ceiling** if it also names a denominator population and ties to a stakeholder's own words.

In [ ]:
RUBRIC = {
    'measurable_threshold': 'Has a number: %, count, time, or $ (not an adjective)',
    'named_population':      'States WHICH cases (Type-A, the golden set, a subset)',
    'testable_day_one':      'You could write the test today and at handover',
    'from_the_client':       'Traces to a stakeholder need (e.g. the Head of Ops: zero Reg E breaches)',
    'guardrail_present':     'At least one do-no-harm criterion (agent-caused breaches = 0)',
}

def grade_criteria_set(criteria):
    measurable = [c for c in criteria if score_criterion(c).verdict == 'MEASURABLE']
    has_guardrail = any(re.search(r'\b(0|zero)\b', c, re.I) and
                        re.search(r'breach|error|leak|harm', c, re.I) for c in criteria)
    floor_pass = len(measurable) >= 4
    return {
        'total': len(criteria),
        'measurable': len(measurable),
        'has_guardrail': has_guardrail,
        'FLOOR_PASS (>=4 measurable)': floor_pass,
    }

show('THE EVAL-ANCHORING RUBRIC')
for k, v in RUBRIC.items():
    print(f'  - {k:<22} {v}')
print()

my_criteria = [
    'Auto-triage >=80% of Type-A disputes with >=95% citation accuracy on the 150-case golden set',
    'Agent-caused Reg E clock breaches = 0 across all disputes',
    'Provisional-credit errors <= 2% on the 150-case golden set (at or below the human baseline)',
    'Hard cases (confidence below threshold) routed to a human within 1 hour on all flagged disputes',
]
print(json.dumps(grade_criteria_set(my_criteria), indent=2))
expect(grade_criteria_set(my_criteria)['FLOOR_PASS (>=4 measurable)'], 'sample criteria set passes the M0 floor')

## 4. Stakeholder / RACI builder

**RACI** assigns, for one decision, who is **R**esponsible (does the work), **A**ccountable (owns the outcome — exactly one person), **C**onsulted, and **I**nformed.

The lesson: **the org chart is not the power map.** The people who can kill the pilot are often not the people who signed the contract. At Meridian, the CISO holds a **hard veto** on data, and the front-line analysts hold a **quiet veto** — they route around tools they dislike. Neither signed anything. "**Data is political**" — at a bank, getting access is a negotiation across people, not an API call (the documented Palantir engagement reality).

In [ ]:
class Veto(Enum):
    NONE = 'none'
    HARD = 'hard'          # can stop it dead (CISO on data)
    QUIET = 'quiet'        # kills it silently (users route around a bad tool)
    BANDWIDTH = 'bandwidth' # blocks by being unavailable (overloaded data lead)

@dataclass
class Stakeholder:
    name: str
    role: str
    raci: str               # one of R / A / C / I (or combos like 'C/R')
    veto: Veto
    wants: str
    how_we_lose_them: str
    on_org_chart: bool = True

def build_raci(stakeholders, decision):
    accountable = [s for s in stakeholders if 'A' in s.raci.split('/')]
    problems = []
    if len(accountable) != 1:
        problems.append(f'RACI needs exactly ONE Accountable, found {len(accountable)}')
    hidden_vetoes = [s for s in stakeholders if s.veto != Veto.NONE and not s.on_org_chart]
    if not hidden_vetoes:
        problems.append('No off-org-chart veto found - you probably missed the front-line users')
    return {'decision': decision, 'accountable': accountable, 'hidden_vetoes': hidden_vetoes,
            'problems': problems}

print('Stakeholder model defined.')

In [ ]:
meridian = [
    Stakeholder('Whitfield', 'CIO', 'A', Veto.NONE,
                'a board-ready demo, fast', 'over-engineering; showing a diagram not a number'),
    Stakeholder('the Head of Ops', 'Head of Disputes Ops', 'C/R', Veto.QUIET,
                'stop missing Reg E deadlines', 'pitching a solution before listening'),
    Stakeholder('Raman', 'CISO', 'C', Veto.HARD,
                'nothing breaks; controls hold', 'asking for production data too early'),
    Stakeholder('Okafor', 'Data team lead', 'C/I', Veto.BANDWIDTH,
                'to be left alone', 'needing warehouse work before we have proven value'),
    Stakeholder('front-line analysts', 'Dispute analysts', 'R', Veto.QUIET,
                'fewer interruptions, less busywork', 'a tool that makes their day worse',
                on_org_chart=False),
]

result = build_raci(meridian, decision='The Meridian dispute pilot succeeds')
show('STAKEHOLDER / RACI MAP', 'Decision: ' + result['decision'])
print(f"{'NAME':<28}{'ROLE':<22}{'RACI':<6}{'VETO':<11}WANTS")
print('-' * 95)
for s in meridian:
    flag = '' if s.on_org_chart else '  (NOT on org chart)'
    print(f'{s.name:<28}{s.role:<22}{s.raci:<6}{s.veto.value:<11}{s.wants}{flag}')
print()
print('Accountable:', [s.name for s in result['accountable']])
print('Hidden vetoes (off org chart):', [s.name for s in result['hidden_vetoes']])
expect(result['problems'] == [], 'RACI is well-formed (one Accountable, a hidden veto found)')

## 5. Integration inventory

The **integration inventory** lists every system the agent must touch, with one twist that separates FDEs from app developers: **rate access difficulty along two axes — technical and political.** The hardest system to reach is almost never the most technically complex; it's the most politically guarded.

At Meridian the **core banking** system is a plain REST API (technically easy) you will spend a *month* getting access to (politically hardest) — it's the crown jewels and the CISO owns the gate. Budget for the politics: the documented Palantir reality was weeks of data-access negotiation per week of value demo. The `identity` field captures the W16 question early: can the agent act *as* a specific user?

In [ ]:
@dataclass
class IntegrationTarget:
    system: str
    data: str
    protocol: str
    owner: str
    technical_effort: int   # 1 (easy) .. 5 (hard)
    political_difficulty: int  # 1 (easy) .. 5 (hard)
    identity_note: str

    @property
    def access_risk(self):
        # political dominates - that's the whole lesson
        return self.political_difficulty * 2 + self.technical_effort

def build_inventory(targets):
    ranked = sorted(targets, key=lambda t: t.access_risk, reverse=True)
    return ranked

inventory = [
    IntegrationTarget('Core banking (REST over mainframe)', 'transactions, dispute records',
                      'REST/JSON', 'the CISO (CISO gate)', technical_effort=2,
                      political_difficulty=5, identity_note='must act as the analyst (OBO) - W16'),
    IntegrationTarget('ServiceNow', 'tickets, duplicate disputes', 'REST/JSON',
                      'Ops IT', technical_effort=3, political_difficulty=2,
                      identity_note='service account; per-user scope later'),
    IntegrationTarget('SharePoint policy docs', 'Reg E / GLBA policies (RAG source)',
                      'Graph API / export', 'Compliance', technical_effort=2,
                      political_difficulty=2, identity_note='read-only; one chunk is poisoned (W16)'),
    IntegrationTarget('Data warehouse (Azure)', '18 months of history (W19)', 'SQL / PostgREST',
                      'the data lead (overloaded)', technical_effort=4, political_difficulty=3,
                      identity_note='read-only role; W19 text-to-SQL target'),
    IntegrationTarget('Identity provider (SSO / OIDC)', 'identity / who is asking', 'OIDC / OAuth2',
                      'the CISO', technical_effort=3, political_difficulty=4,
                      identity_note='the agent needs its own identity (OAuth2 on-behalf-of) - W16'),
]

ranked = build_inventory(inventory)
show('INTEGRATION INVENTORY (ranked by access risk = political*2 + technical)')
print(f"{'SYSTEM':<40}{'TECH':<6}{'POL':<6}{'RISK':<6}OWNER")
print('-' * 80)
for t in ranked:
    print(f'{t.system:<40}{t.technical_effort:<6}{t.political_difficulty:<6}{t.access_risk:<6}{t.owner}')
print()
print('Hardest to access:', ranked[0].system)
print('Why:', 'political (' + str(ranked[0].political_difficulty) + '/5), not technical ('
      + str(ranked[0].technical_effort) + '/5) - it is the crown jewels behind the CISO gate.')
expect('Core banking' in ranked[0].system, 'core banking ranks hardest to access')

## 6. The decomposition drill — worked example (city 911 response times)

The **case-study (decomposition) round** — popularised by Palantir, now run in some form by most FDE employers — is the interview's signature filter. Given a vague, large problem, the interviewer watches whether you **scope it** or **panic-solve it**. The move is **Clarify -> Decompose -> Scope**, the same muscle as the discovery workshop, minus the friendly client.

Worked prompt: *"A city's 911 response times are too slow. Decompose it."* The classic trap is answering "add more ambulances" — a solution to an undiagnosed problem. Here is the disciplined path.

In [ ]:
# Step 1 - CLARIFY: questions that shrink the problem before you touch it.
CLARIFYING_QUESTIONS = [
    'Slow where - call pickup, triage, dispatch, or travel to scene?',
    'Which incident types - all of them, or a specific class (cardiac, fire, crime)?',
    '"Too slow" versus what target - is there an SLA we are breaching, and by how much?',
    'Slow everywhere, or concentrated in certain districts / times of day?',
    'What can we actually change - staffing, routing software, station locations?',
]
show('STEP 1 - CLARIFY (shrink the problem first)')
for q in CLARIFYING_QUESTIONS:
    print('  ? ' + q)

In [ ]:
# Step 2 - DECOMPOSE: break the end-to-end time into named, measurable segments.
# Each segment is a different problem with a different owner and a different fix.
RESPONSE_FUNNEL = [
    ('Call answer',  'caller dials -> dispatcher picks up',      'call-center staffing / IVR'),
    ('Triage',       'classify the emergency, get the address',  'protocol + dispatcher training'),
    ('Dispatch',     'select + assign the right unit',           'routing software / unit availability'),
    ('Travel',       'unit leaves -> arrives on scene',          'station placement / traffic / fleet'),
    ('On-scene',     'arrival -> stabilized',                    'crew capability (out of scope for v1)'),
]
show('STEP 2 - DECOMPOSE the response funnel')
print(f"{'SEGMENT':<14}{'WHAT':<42}OWNER / LEVER")
print('-' * 90)
for seg, what, lever in RESPONSE_FUNNEL:
    print(f'{seg:<14}{what:<42}{lever}')
print()
print('Each segment is measurable independently. You cannot fix what you have not split.')

In [ ]:
# Step 3 - SCOPE: pick the dominant, fixable segment. Attach a measurable target.
# Suppose data shows where the time actually goes:
segment_share = {
    'Call answer': 0.10,
    'Triage':      0.10,
    'Dispatch':    0.45,   # the dominant, fixable bottleneck
    'Travel':      0.30,
    'On-scene':    0.05,
}

dominant = max(segment_share, key=segment_share.get)
show('STEP 3 - SCOPE')
for seg, share in sorted(segment_share.items(), key=lambda kv: kv[1], reverse=True):
    bar = '#' * int(share * 40)
    print(f'  {seg:<14}{share*100:5.0f}%  {bar}')
print()
print(f'Dominant fixable segment: {dominant} ({segment_share[dominant]*100:.0f}% of delay).')
print('SCOPED v1: cut DISPATCH time, not the whole funnel.')
print('Eval anchor: reduce median dispatch time by >=30% on cardiac calls in the 2 worst districts,')
print('             measured over 30 days. Ignore travel/on-scene for v1.')
print()
print('Note: "add more ambulances" attacks TRAVEL (30%), not the dominant DISPATCH segment (45%) -')
print('the solution-jump would have spent the budget on the wrong problem.')
expect(dominant == 'Dispatch', 'the drill scopes the dominant fixable segment, not the obvious one')

### The pattern, generalized

`Clarify -> Decompose -> Scope` is the FDE's core move in **both** rooms:

| | Discovery workshop (W13 sim) | Decomposition interview |
|---|---|---|
| Clarify | "Show me how you do this today" | "Slow where? vs what target?" |
| Decompose | the real workflow, step by step | the response funnel, segment by segment |
| Scope | the eval-anchored success criteria | the dominant fixable segment + a metric |

Same skill. The interview just removes the friendly client and the 90 minutes.

## 6.5 The throwaway POC — rapid prototyping in discovery

The decomposition drill above ends with a scoped, measurable target. Before you commit a milestone to it, an FDE runs one more move: the **throwaway POC** — the only kind of *building* that belongs in the scoping week.

The discipline has three rules:

1. **Timebox it.** Minutes-to-days, never weeks. The POC answers a question; it does not accrete features. When the clock runs out, you have your answer either way.
2. **One assumption per POC.** Pick the single riskiest assumption in your scope and attack only that. A POC that tests two things proves neither.
3. **Delete it after.** The POC's output is a *yes/no plus what you learned* — recorded in `DECISIONS.md` — not code. The moment a POC ships, it becomes demo-ware in production (the W15 war story). Build the real thing fresh, against evals.

Below: a complete POC in ~15 lines against one Meridian assumption — *"can we classify dispute type (Type-A fraud vs Type-B merchant) from the card-network reason code alone, with no model?"* If yes, triage v0 is a lookup, and the LLM budget goes to the genuinely hard cases.


In [ ]:
# THE THROWAWAY POC — timeboxed, one assumption, built to be deleted.
# Assumption under test: dispute type (A = fraud / B = merchant-quality) is
# derivable from the network reason code alone — no model needed for triage v0.
SAMPLE = [  # (dispute_id, network reason code, analyst ground-truth type)
    (4417, "10.4", "A"), (4421, "13.1", "B"), (4433, "10.1", "A"),
    (4438, "13.3", "B"), (4442, "12.6", "B"), (4451, "10.4", "A"),
    (4460, "13.2", "B"), (4465, "11.3", "?"),   # 11.x = authorization — ambiguous
]

def classify(reason_code):
    # Card-network convention: 10.x = fraud, 12.x processing, 13.x consumer dispute.
    major = reason_code.split(".")[0]
    return "A" if major == "10" else ("B" if major in {"12", "13"} else "?")

hits = 0
for did, rc, truth in SAMPLE:
    got = classify(rc)
    hits += (got == truth)
    print(f"dispute {did}  code {rc}  ->  Type-{got:1}   truth Type-{truth:1}   "
          f"{'ok' if got == truth else 'MISS'}")
print(f"\nPOC verdict: {hits}/{len(SAMPLE)} from a 3-line rule -> the assumption HOLDS "
      "for the easy majority;")
print("the 11.x authorization codes are exactly the ambiguous slice a human (or model) must own.")
# Question answered. Log the verdict in DECISIONS.md, then DELETE this cell's code
# as a design: the production triage agent (W15) is built fresh, against evals.


## 7. Self-check for the discovery-workshop artifact

`score_process_map()` checks the floor for the process map you build in the live simulation: it must capture at least one **shadow process** (a step outside the official systems), touch at least three source systems, and end on a measurable success definition. A map that only lists the clean demo path fails — that's the exact mistake the three prior vendors made.

In [ ]:
KNOWN_SYSTEMS = {'core banking', 'core_banking', 'servicenow', 'sharepoint',
                 'policy', 'card network', 'card_network', 'warehouse'}

def score_process_map(steps, shadow_flags, success_definition):
    """steps: list[str]; shadow_flags: list[bool] (same length); success_definition: str."""
    assert len(steps) == len(shadow_flags), 'each step needs a shadow flag'
    has_shadow = any(shadow_flags)
    systems_hit = {s for st in steps for s in KNOWN_SYSTEMS if s in st.lower()}
    enough_systems = len(systems_hit) >= 3
    success_measurable = score_criterion(success_definition).verdict in ('MEASURABLE', 'PARTIAL (has a number, missing population)')
    floor = has_shadow and enough_systems and success_measurable
    return {
        'has_shadow_step': has_shadow,
        'systems_touched': sorted(systems_hit),
        'enough_systems (>=3)': enough_systems,
        'success_is_measurable': success_measurable,
        'FLOOR_PASS': floor,
    }

# A good cohort's map (includes the shadow spreadsheet):
good_steps = [
    'Classify Type-A vs Type-B (judgment)',
    'Pull the transaction in core banking',
    'Pull 90 days of activity from core banking',
    'Check ServiceNow for an existing ticket (duplicate check)',
    'Check the analyst merchant-pattern spreadsheet',  # the shadow step
    'Check the SharePoint policy doc for the Reg E rule',
    'Decide provisional credit',
    'Route the hard case to a human with evidence gathered',
]
good_shadow = [False, False, False, False, True, False, False, False]
good_success = 'Zero Reg E breaches on all disputes; humans handle the hard 20% of cases'

show('SELF-CHECK: a GOOD process map')
print(json.dumps(score_process_map(good_steps, good_shadow, good_success), indent=2))

# A demo-path-only map (what the failed vendors produced):
bad_steps = ['Classify the dispute', 'Look up the transaction in core banking', 'Resolve it']
bad_shadow = [False, False, False]
bad_success = 'Make disputes faster'
print()
show('SELF-CHECK: a DEMO-PATH-ONLY map (fails)')
print(json.dumps(score_process_map(bad_steps, bad_shadow, bad_success), indent=2))

expect(score_process_map(good_steps, good_shadow, good_success)['FLOOR_PASS'], 'good map passes the floor')
expect(not score_process_map(bad_steps, bad_shadow, bad_success)['FLOOR_PASS'], 'demo-path map fails the floor')

## 8. Putting it together — a starter M0 charter

This cell writes a skeleton charter to `m0_charter/` using everything above, so you can see the five artifacts as files. Replace the contents with your own work for the graded milestone (`lab/milestone-M0-charter.md`).

In [ ]:
def write_file(name, content):
    p = os.path.join(WORKDIR, name)
    os.makedirs(os.path.dirname(p), exist_ok=True) if os.path.dirname(p) else None
    with open(p, 'w') as f:
        f.write(content)
    return p

problem = (
    '# Problem statement\n\n'
    'Meridian doubled debit-card dispute volume after a portfolio acquisition. 35 analysts '
    'triage by hand at a 24-minute average; ~12% of disputes breach the Reg E 10-business-day '
    'clock, and provisional-credit errors lose real money. The mechanism is integration friction '
    '(evidence scattered across >=3 systems) plus tacit-knowledge routing (seniors interrupted to '
    'answer juniors). The constraint is the non-negotiable Reg E 10-business-day SLA.\n'
)

criteria_md = '# Eval-anchored success criteria\n\n' + '\n'.join('- ' + c for c in my_criteria) + '\n'

stakeholders_md = '# Stakeholder / RACI map\n\nDecision: the pilot succeeds (Accountable: Whitfield, CIO)\n\n'
stakeholders_md += '\n'.join(
    f'- **{s.name}** ({s.role}) - RACI {s.raci}, veto={s.veto.value}, wants: {s.wants}'
    for s in meridian) + '\n'

inv_md = '# Integration inventory\n\n'
inv_md += '\n'.join(
    f'- **{t.system}** - {t.data} via {t.protocol}; owner {t.owner}; '
    f'tech {t.technical_effort}/5, political {t.political_difficulty}/5; {t.identity_note}'
    for t in ranked) + '\n'

paths = [
    write_file('charter/problem-statement.md', problem),
    write_file('charter/success-criteria.md', criteria_md),
    write_file('charter/stakeholders.md', stakeholders_md),
    write_file('charter/integrations.md', inv_md),
    os.path.join(WORKDIR, 'DECISIONS.md'),  # already written in section 2
]

show('M0 CHARTER SKELETON WRITTEN')
for p in paths:
    print('  ' + p, '(exists)' if os.path.exists(p) else '(MISSING)')
for p in paths:
    expect(os.path.exists(p), 'artifact exists: ' + os.path.basename(p))

print()
print('All five M0 artifacts present. Replace with your own work for grading.')

## 9. Glossary — every named term from the slides

| Term | Meaning |
|---|---|
| **FDE (Forward-Deployed Engineer)** | An engineer embedded with the client who scopes, builds against real systems, proves value, and hands over. Closes the deployment gap NANDA measured. |
| **FDE lifecycle** | Scope -> Build -> Deploy -> Operate. This course's spine; each week is one stage. |
| **The 95% problem** | MIT NANDA: ~95% of organizations get zero measurable return from GenAI pilots; the bottleneck is deployment, not models. |
| **Whole-game demo** | fast.ai pedagogy: show the finished system first, then earn it. W13 hour 1. |
| **Discovery workshop** | A structured interview that extracts the real workflow, led by "show me how you do this today." |
| **"Show me how you do this today"** | The FDE's core discovery sentence: surfaces the real workflow, disarms skeptics, finds shadow processes. |
| **Shadow process** | A workflow people rely on that lives outside the official systems (e.g. the senior analyst's merchant-pattern spreadsheet). |
| **Stakeholder map** | Everyone who can authorize, block, or veto - including off-org-chart vetoes. |
| **RACI** | Responsible / Accountable (exactly one) / Consulted / Informed, per decision. |
| **"Data is political"** | At an enterprise, data access is a negotiation across people, not an API call (Palantir engagement reality). |
| **Eval-anchored** | Success written as a measurable, testable criterion ([metric] [threshold] [population]), scoped at discovery. |
| **Acceptance criteria** | The measurable conditions that define "done"; here, they ARE the eval anchors. |
| **Integration inventory** | Every system to touch, rated by technical AND political access difficulty. |
| **Identity (in inventory)** | Can the agent act as a specific user? Seed of W16 RBAC / OAuth2 on-behalf-of / agent identity. |
| **DECISIONS.md** | ADR-lite log: Decision / Why / Alternatives / Owner / Revisit. Scope-creep armor. |
| **Decomposition round** | The case-study interview round (Palantir popularised it; now industry-wide): Clarify -> Decompose -> Scope before solving. |
| **Reg E** | US regulation putting a hard 10-business-day clock on payment disputes; the engagement's teachable SLA. |
| **Type-A / Type-B dispute** | A = fraud/unauthorized; B = merchant/quality. The RBAC scoping axis and a triage decision. |

---

**You scoped it.** Next week you price it: M1 - SoW, pricing, and the reference architecture. The CFO wants a number by Friday.